# M04_BZ_SCC — Border Zone & Corridor Basic Statistics

Basic geometric statistics for the Border Zone (BZ) and scar corridors (SCC) across the cohort.

| Section | Contents |
|---|---|
| 1 | Setup: imports, scan, tissue availability |
| 2 | Border Zone statistics: area, BZ/scar ratio |
| 3 | Corridor statistics: detection rate, count, area, centerline length |
| 4 | Summary table |


## Section 1 — Setup

### 1.1 Imports and Configuration


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
from tqdm import tqdm


def find_project_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / 'lv-scar-segmentation'):
            if (candidate / 'src' / 'data_loading.py').exists():
                return candidate.resolve()
    raise FileNotFoundError('Could not locate lv-scar-segmentation.')


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for mod in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[mod]

from src.data_loading import scan

HD_ROOT = r'F:/RM_TEKNON_DEVELOP'
RESULTS = ROOT / 'results'
OUT = RESULTS / 'M04_BZ_SCC'
OUT.mkdir(parents=True, exist_ok=True)

pv.set_jupyter_backend('static')
plt.rcParams['figure.dpi'] = 120
print(f'Project root: {ROOT}')
print('Config done.')


### 1.2 Data Loading and Tissue Availability


In [ ]:
cases = scan(HD_ROOT)
cases = [c for c in cases if c.lv_dir is not None]
print(f'{len(cases)} cases with LV directory')

def _has(case, key):
    p = case.tissue_surfaces.get(key)
    return p is not None and Path(str(p)).exists()

availability = {
    'border_zone':                  sum(_has(c, 'border_zone') for c in cases),
    'scar':                         sum(_has(c, 'scar') for c in cases),
    'corridors':                    sum(_has(c, 'corridors') for c in cases),
    'corridor_labels':              sum(_has(c, 'corridor_labels') for c in cases),
    'corridor_centerlines_unified': sum(_has(c, 'corridor_centerlines_unified') for c in cases),
    'corridor_bz':                  sum(_has(c, 'corridor_bz') for c in cases),
}

avail_df = pd.DataFrame({
    'file': list(availability.keys()),
    'n_found': list(availability.values()),
    'pct': [100 * v / len(cases) for v in availability.values()],
})
print(avail_df.to_string(index=False))


## Section 2 — Border Zone Statistics

### 2.1 BZ Area and BZ/Scar Ratio


In [ ]:
def mesh_area_cm2(path):
    """Surface area in cm² (mesh units assumed mm)."""
    if path is None or not Path(str(path)).exists():
        return np.nan
    return pv.read(str(path)).area / 100.0


bz_rows = []
for c in tqdm(cases, desc='BZ stats'):
    bz_area   = mesh_area_cm2(c.tissue_surfaces.get('border_zone'))
    sc_area   = mesh_area_cm2(c.tissue_surfaces.get('scar'))
    ratio     = bz_area / sc_area if (np.isfinite(sc_area) and sc_area > 0) else np.nan
    bz_rows.append({
        'pid':            c.patient_id,
        'bz_area_cm2':    bz_area,
        'scar_area_cm2':  sc_area,
        'bz_scar_ratio':  ratio,
    })

bz_df = pd.DataFrame(bz_rows)
bz_df.to_csv(OUT / 'bz_stats.csv', index=False)
bz_df.describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(bz_df['bz_area_cm2'].dropna(), bins=15, edgecolor='k')
axes[0].set(xlabel='BZ area (cm²)', ylabel='N patients', title='Border Zone Area')

axes[1].hist(bz_df['scar_area_cm2'].dropna(), bins=15, edgecolor='k', color='firebrick')
axes[1].set(xlabel='Scar area (cm²)', ylabel='N patients', title='Scar Area')

axes[2].hist(bz_df['bz_scar_ratio'].dropna(), bins=15, edgecolor='k', color='darkorange')
axes[2].set(xlabel='BZ / Scar ratio', ylabel='N patients', title='BZ-to-Scar Ratio')

fig.tight_layout()
fig.savefig(OUT / 'bz_distributions.png', dpi=150)
plt.show()


## Section 3 — Corridor Statistics

### 3.1 Corridor Count per Patient

Corridor count read from the Labels VTK scalar array (unique positive label IDs = distinct corridors).


In [ ]:
def count_corridors(path):
    """Count unique corridor IDs from a Labels VTK (positive label values)."""
    if path is None or not Path(str(path)).exists():
        return np.nan
    m = pv.read(str(path))
    for arrays in (m.cell_data, m.point_data):
        for name in arrays.keys():
            arr = np.asarray(arrays[name]).ravel().astype(float)
            ids = np.unique(arr[np.isfinite(arr)].astype(int))
            ids = ids[ids > 0]
            if len(ids) > 0:
                return int(len(ids))
    return np.nan


def centerline_length_mm(path):
    """Total centerline length in mm (sum of consecutive point distances)."""
    if path is None or not Path(str(path)).exists():
        return np.nan
    m = pv.read(str(path))
    if m.n_points < 2:
        return np.nan
    pts = np.asarray(m.points)
    return float(np.sum(np.linalg.norm(np.diff(pts, axis=0), axis=1)))


corr_rows = []
for c in tqdm(cases, desc='Corridor stats'):
    corr_path   = c.tissue_surfaces.get('corridors')
    labels_path = c.tissue_surfaces.get('corridor_labels')
    cl_path     = c.tissue_surfaces.get('corridor_centerlines_unified')
    cbz_path    = c.tissue_surfaces.get('corridor_bz')
    has_corr    = _has(c, 'corridors')
    corr_rows.append({
        'pid':               c.patient_id,
        'has_corridors':     has_corr,
        'corridor_area_cm2': mesh_area_cm2(corr_path),
        'corridor_bz_area_cm2': mesh_area_cm2(cbz_path),
        'n_corridors':       count_corridors(labels_path),
        'cl_length_mm':      centerline_length_mm(cl_path),
    })

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(OUT / 'corridor_stats.csv', index=False)

n_with = corr_df['has_corridors'].sum()
print(f'Corridor data: {n_with}/{len(corr_df)} cases ({100*n_with/len(corr_df):.1f}%)')
corr_df[corr_df['has_corridors']].describe().round(2)


### 3.2 Corridor Distributions


In [ ]:
sub = corr_df[corr_df['has_corridors']].copy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

n_max = int(sub['n_corridors'].dropna().max()) if sub['n_corridors'].notna().any() else 10
axes[0].hist(sub['n_corridors'].dropna(), bins=range(1, n_max + 2), edgecolor='k', color='steelblue')
axes[0].set(xlabel='N corridors', ylabel='N patients', title='Corridors per Patient')

axes[1].hist(sub['corridor_area_cm2'].dropna(), bins=15, edgecolor='k', color='steelblue')
axes[1].set(xlabel='Corridor area (cm²)', ylabel='N patients', title='Total Corridor Area')

axes[2].hist(sub['corridor_bz_area_cm2'].dropna(), bins=15, edgecolor='k', color='mediumseagreen')
axes[2].set(xlabel='Corridor BZ area (cm²)', ylabel='N patients', title='Corridor Border Zone Area')

axes[3].hist(sub['cl_length_mm'].dropna(), bins=15, edgecolor='k', color='orchid')
axes[3].set(xlabel='CL length (mm)', ylabel='N patients', title='Unified Centerline Length')

fig.tight_layout()
fig.savefig(OUT / 'corridor_distributions.png', dpi=150)
plt.show()


## Section 4 — Summary Table


In [ ]:
merged = bz_df.merge(
    corr_df.drop(columns='has_corridors'),
    on='pid', how='outer'
)
merged.to_csv(OUT / 'bz_corridor_summary.csv', index=False)

cols = ['bz_area_cm2', 'scar_area_cm2', 'bz_scar_ratio',
        'corridor_area_cm2', 'corridor_bz_area_cm2', 'n_corridors', 'cl_length_mm']
summary = merged[cols].describe().round(2)
print(summary.to_string())
print(f'\nSaved to {OUT}')
